# Urgency Tagging with LSTM Model

Mengimplementasikan model **LSTM (Long Short-Term Memory)** untuk klasifikasi tingkat urgensi komplain konsumen: **High**, **Medium**, dan **Low**.

## Alur Pipeline
1. Load dataset ter-lemmatisasi (`cleaned_complaints.csv`)
2. Pembagian data stratified (Train 70%, Val 15%, Test 15%)
3. **[EDA]** Analisis parameter tokenisasi (`max_words` & `max_len`)
4. Label encoding kelas target
5. Tokenisasi teks & padding sekuens
6. Penanganan kelas imbalanced dengan *class weight*
7. Definisi arsitektur model LSTM
8. Pelatihan dengan Early Stopping & ModelCheckpoint
9. Visualisasi Loss & Accuracy
10. Evaluasi (Classification Report & Confusion Matrix)
11. Simpan artefak model ke `models/`

## Tahap 1 — Setup: Import Library & Konfigurasi Environment

- Deteksi lingkungan (Google Colab / lokal) dan atur `PROJECT_ROOT`
- Import seluruh library yang dibutuhkan
- Set global random seed (`SEED=42`) untuk reprodusibilitas

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pickle

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/lentera-analytics-hub/lentera-ml-research'
    if not os.path.exists(PROJECT_ROOT):
        PROJECT_ROOT = '/content/drive/MyDrive/lentera-ml-research'
    os.chdir(PROJECT_ROOT)
else:
    notebook_dir = os.getcwd()
    PROJECT_ROOT = os.path.abspath(os.path.join(notebook_dir, '..')) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.features import split_data, extract_lstm_sequences

import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.text import Tokenizer

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"Project root: {PROJECT_ROOT}")

## Tahap 2 — Data: Load, Validasi & Split

### 1. Load dan Validasi Dataset

- Baca `cleaned_complaints.csv` dari `data/processed/`
- Hapus baris dengan nilai kosong pada kolom `Cleaned_Narrative` atau `Urgency`
- Tampilkan distribusi kelas sebagai sanity check

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'cleaned_complaints.csv')
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['Cleaned_Narrative', 'Urgency'])

print(f"Shape dataset: {df.shape}")
print("\nDistribusi kelas Urgency:")
print(df['Urgency'].value_counts())

### 2. Pembagian Data (Data Splitting)

- Split stratified: **Train 70%**, **Validation 15%**, **Test 15%**
- Menggunakan `split_data` dari `src.features` untuk konsistensi lintas notebook

In [ ]:
df_train, df_val, df_test = split_data(
    df,
    text_col='Cleaned_Narrative',
    target_col='Urgency'
)

## Tahap 3 — EDA Parameter Tokenisasi

### 3. Analisis Parameter Tokenisasi

Menentukan `max_words` dan `max_len` secara empiris dari data latih — bukan nilai hardcoded.

- **`max_words`**: terlalu besar → boros memori; terlalu kecil → banyak kata OOV
- **`max_len`**: terlalu besar → padding berlebihan; terlalu kecil → konteks terpotong

> ⚠️ Analisis dilakukan **hanya pada `df_train`** untuk menghindari data leakage.

In [ ]:
train_token_lengths = df_train['Cleaned_Narrative'].astype(str).apply(lambda x: len(x.split()))

print(f"Jumlah sampel : {len(train_token_lengths):,}")
print(f"Min           : {train_token_lengths.min()} token")
print(f"Mean          : {train_token_lengths.mean():.1f} token")
print(f"Median (P50)  : {train_token_lengths.median():.0f} token")
print(f"P75           : {train_token_lengths.quantile(0.75):.0f} token")
print(f"P90           : {train_token_lengths.quantile(0.90):.0f} token")
print(f"P95           : {train_token_lengths.quantile(0.95):.0f} token")
print(f"P99           : {train_token_lengths.quantile(0.99):.0f} token")
print(f"Max           : {train_token_lengths.max():,} token")

In [ ]:
p90 = int(train_token_lengths.quantile(0.90))
p95 = int(train_token_lengths.quantile(0.95))
p99 = int(train_token_lengths.quantile(0.99))
median_len = int(train_token_lengths.median())

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(train_token_lengths, bins=80, color='steelblue', alpha=0.75, edgecolor='white', linewidth=0.5)
ax.axvline(median_len, color='green',  linestyle='--', linewidth=1.8, label=f'Median = {median_len}')
ax.axvline(p90,        color='orange', linestyle='--', linewidth=1.8, label=f'P90    = {p90}')
ax.axvline(p95,        color='red',    linestyle='--', linewidth=1.8, label=f'P95    = {p95}')
ax.axvline(p99,        color='purple', linestyle=':',  linewidth=1.8, label=f'P99    = {p99}')
ax.set_title('Distribusi Panjang Token per Teks — Data Latih', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Jumlah Token (kata)', fontsize=12)
ax.set_ylabel('Frekuensi', fontsize=12)
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mticker.MultipleLocator(50))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
_probe_tokenizer = Tokenizer(oov_token="<OOV>")
_probe_tokenizer.fit_on_texts(df_train['Cleaned_Narrative'])

actual_vocab_size = len(_probe_tokenizer.word_index)
word_counts = sorted(_probe_tokenizer.word_counts.values(), reverse=True)
total_words = len(word_counts)
total_token_count = sum(word_counts)

print(f"Vocabulary aktual (data latih): {actual_vocab_size:,} kata unik\n")
print(f"{'max_words':>10}  {'Coverage (%)':>14}  {'Kata Diabaikan':>16}  {'Keputusan':>12}")
print("-" * 58)
for threshold in [3000, 5000, 8000, 10000, 12000, 15000, 20000]:
    if threshold >= total_words:
        coverage, ignored = 100.0, 0
    else:
        coverage = sum(word_counts[:threshold]) / total_token_count * 100
        ignored = total_words - threshold
    verdict = "✅ Sangat baik" if coverage >= 99.0 else "✅ Baik" if coverage >= 97.0 else "⚠️  Cukup" if coverage >= 95.0 else "❌ Kurang"
    print(f"{threshold:>10,}  {coverage:>13.2f}%  {ignored:>16,}  {verdict:>12}")

In [ ]:
p95_raw = train_token_lengths.quantile(0.95)
FINAL_MAX_LEN = int(np.ceil(p95_raw / 25) * 25)

FINAL_MAX_WORDS = None
for threshold in range(1000, actual_vocab_size + 1, 500):
    if sum(word_counts[:threshold]) / total_token_count >= 0.98:
        FINAL_MAX_WORDS = threshold
        break

if FINAL_MAX_WORDS is None:
    FINAL_MAX_WORDS = actual_vocab_size
FINAL_MAX_WORDS = min(FINAL_MAX_WORDS, 20000)

print(f"FINAL max_words = {FINAL_MAX_WORDS:,}  (coverage ≥ 98% token)")
print(f"FINAL max_len   = {FINAL_MAX_LEN}     (P95={p95_raw:.0f}, dibulatkan ke kelipatan 25)")
print(f"Teks terpotong  = {(train_token_lengths > FINAL_MAX_LEN).mean()*100:.1f}%")

## Tahap 4 — Preprocessing: Encoding & Tokenisasi

### 4. Label Encoding Target

- Encode label kategori (`High`, `Medium`, `Low`) menjadi integer menggunakan `LabelEncoder`
- `fit` dilakukan pada data train; `transform` pada val dan test

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(df_train['Urgency'])
y_val_encoded   = label_encoder.transform(df_val['Urgency'])
y_test_encoded  = label_encoder.transform(df_test['Urgency'])

class_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Pemetaan kelas target:", class_mapping)

### 5. Tokenisasi dan Padding Sekuens

- Ubah teks menjadi sekuens integer menggunakan `extract_lstm_sequences`
- Parameter `max_words` dan `max_len` berasal dari hasil EDA (Tahap 3)

In [ ]:
X_train_pad, X_val_pad, X_test_pad, tokenizer = extract_lstm_sequences(
    df_train['Cleaned_Narrative'],
    df_val['Cleaned_Narrative'],
    df_test['Cleaned_Narrative'],
    max_words=FINAL_MAX_WORDS,
    max_len=FINAL_MAX_LEN
)

print(f"X_train_pad : {X_train_pad.shape}")
print(f"X_val_pad   : {X_val_pad.shape}")
print(f"X_test_pad  : {X_test_pad.shape}")
print(f"Vocabulary efektif: {len(tokenizer.word_index):,} kata")

### 6. Penanganan Kelas Imbalanced

- Hitung `class_weight` dengan metode `balanced` untuk menaikkan bobot kelas minoritas
- **Manual tuning**: bobot kelas `Low` diturunkan 25% karena `balanced` cenderung terlalu agresif → model *over-predict* Low
- Rentang penurunan yang disarankan: 20–30%

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weights_dict = dict(enumerate(class_weights))

low_idx = class_mapping.get("Low")
if low_idx is not None:
    _orig_low = class_weights_dict[low_idx]
    class_weights_dict[low_idx] = _orig_low * 0.75
    print(f"Manual tuning kelas Low: {_orig_low:.4f} → {class_weights_dict[low_idx]:.4f} (-25%)")

print("\nBobot kelas final:")
for name, idx in class_mapping.items():
    print(f"  {name:8s} (idx={idx}): {class_weights_dict[idx]:.4f}")

## Tahap 5 — Model: Definisi, Training & Evaluasi

### 7. Definisi Arsitektur Model LSTM

Arsitektur sekuensial dengan lapisan:
- **Embedding** (128 dim): konversi integer → vektor padat
- **Bidirectional LSTM** (128 unit): menangkap konteks dari dua arah; 3 gate + Cell State untuk dependensi jangka panjang
- **Dense** (32 unit, ReLU): representasi laten
- **Dropout** (0.5): regularisasi untuk mencegah overfitting
- **Output** (3 unit, Softmax): distribusi probabilitas per kelas

| Komponen | GRU | LSTM |
|----------|-----|------|
| Layer RNN | `Bidirectional(GRU(128))` | `Bidirectional(LSTM(128))` |
| Gate | 2 | 3 |
| State | Hidden state | Hidden state + Cell state |
| ~Parameter | 198K | 230K |

In [ ]:
vocab_size    = FINAL_MAX_WORDS
embedding_dim = 128
max_len       = FINAL_MAX_LEN
num_classes   = len(class_mapping)

model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0)),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### 8. Pelatihan Model

- **Epochs**: 50
- **Early Stopping**: berhenti jika `val_loss` tidak membaik selama **7 epoch** berturut-turut, kembalikan bobot terbaik
- **ModelCheckpoint**: simpan model terbaik otomatis berdasarkan `val_loss`

In [ ]:
MODEL_SAVE_PATH = os.path.join(PROJECT_ROOT, 'models', 'lstm_urgency_model.keras')

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train_pad,
    y_train_encoded,
    validation_data=(X_val_pad, y_val_encoded),
    epochs=50,
    batch_size=64,
    class_weight=class_weights_dict,
    callbacks=[early_stopping, model_checkpoint]
)

### 9. Visualisasi Riwayat Pelatihan

- Plot kurva **Loss** dan **Accuracy** untuk train vs validation
- Berguna untuk mendeteksi gejala *underfitting* atau *overfitting*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss',     marker='o')
axes[0].plot(history.history['val_loss'], label='Val Loss',       marker='o')
axes[0].set_title('Training & Validation Loss (LSTM)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'],     label='Train Accuracy', marker='o')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   marker='o')
axes[1].set_title('Training & Validation Accuracy (LSTM)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'models', 'lstm_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

### 10. Evaluasi Model

- Prediksi pada **test set** (data yang belum pernah dilihat model)
- **Classification Report**: Precision, Recall, F1-Score per kelas
- **Confusion Matrix**: visualisasi distribusi prediksi vs aktual

In [ ]:
y_pred_proba = model.predict(X_test_pad, batch_size=64)
y_pred = np.argmax(y_pred_proba, axis=1)

test_loss, test_accuracy = model.evaluate(X_test_pad, y_test_encoded, batch_size=64, verbose=0)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")

print(classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_encoded,
    y_pred,
    display_labels=label_encoder.classes_,
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix — LSTM Model (Test Set)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'models', 'lstm_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## Tahap 6 — Simpan Artefak

### 11. Simpan Tokenizer & Label Encoder

- Tokenizer dan label encoder disimpan bersama model agar dapat digunakan kembali saat **inferensi**
- Format: `.pkl` (pickle) untuk tokenizer dan label encoder; `.keras` untuk model

In [ ]:
TOKENIZER_SAVE_PATH    = os.path.join(PROJECT_ROOT, 'models', 'lstm_tokenizer.pkl')
LABEL_ENCODER_SAVE_PATH = os.path.join(PROJECT_ROOT, 'models', 'lstm_label_encoder.pkl')

with open(TOKENIZER_SAVE_PATH, 'wb') as f:
    pickle.dump(tokenizer, f)

with open(LABEL_ENCODER_SAVE_PATH, 'wb') as f:
    pickle.dump(label_encoder, f)

print("Artefak tersimpan:")
print(f"  Model       : models/lstm_urgency_model.keras")
print(f"  Tokenizer   : models/lstm_tokenizer.pkl")
print(f"  LabelEncoder: models/lstm_label_encoder.pkl")
print(f"  Plot Loss   : models/lstm_training_history.png")
print(f"  Plot CM     : models/lstm_confusion_matrix.png")
print(f"\nParameter tokenisasi: max_words={FINAL_MAX_WORDS:,}, max_len={FINAL_MAX_LEN}")